# Day 1 - Explore the ticket data
We only look at the data here. Nothing is changed on disk.

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

df = pd.read_csv("../data/raw/tickets.csv")

## 1. Shape, columns, sample rows

In [2]:
print("Shape (rows, columns):", df.shape)
print()
print(list(df.columns))

Shape (rows, columns): (28587, 16)

['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


In [3]:
df.sample(5, random_state=1)

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
482,Billing Statement,"Dear Customer Support Team,\n\nI am reaching out to highlight an ongoing problem with the billing system concerning ...",Thank you for informing us about the billing processing problem caused by the synchronization malfunction in the pro...,Problem,Billing and Payments,low,en,51,Billing,Hardware,Issue,Investigation,NaN,NaN,NaN,NaN
1916,Firewall Issue,"The USB hub connected to the firewall caused disconnections and disrupted data analysis, potentially due to firmware...",Received an email regarding the firewall issue; the disconnections interfere with data analysis. To continue trouble...,Problem,Technical Support,high,de,52,Firewall,Network,Disruption,Recovery,Firmware,NaN,NaN,NaN
242,Revision of Return and Exchange Workflows,"Sehr geehrtes Kundenservice-Team,\n\nich hoffe, diese Nachricht erreicht Sie wohlbehalten. Ich möchte um eine Aktual...",Vielen Dank für die ausführliche Anfrage zur Aktualisierung der Rückgabe- und Umtauschprozesse. Wir werden einen umf...,Change,Returns and Exchanges,medium,de,51,Product,Workflow,Feedback,Customer Service,NaN,NaN,NaN,NaN
13462,Unterstützung für das Projektmanagement,"Sehr geehrte Kundenservice, ich kontaktiere Sie im Zusammenhang mit der Integration des WooCommerce in das TYPO3-Pro...",Wir werden den Antrag bearbeiten und Ihnen bald mit Ratschlägen zur Integration von WooCommerce in TYPO3 kontaktieren.,Request,IT Support,medium,de,400,Feature,Documentation,Feedback,IT,Tech Support,NaN,NaN,NaN
5625,Inquiry About Digital Marketing Solutions Today,I am interested in learning about products related to digital marketing services. Could you please provide detailed ...,"<name>,\nThank you for your interest in our digital marketing solutions. I would be happy to provide detailed billin...",Request,Billing and Payments,medium,en,52,Sales,Lead,Payment,Billing,NaN,NaN,NaN,NaN


## 2. Language split, then keep English only

In [4]:
print(df["language"].value_counts())
df = df[df["language"] == "en"].copy()
print("English rows kept:", len(df))

language
en    16338
de    12249
Name: count, dtype: int64
English rows kept: 16338


## 3. Empty values and duplicates (English only)

In [5]:
empty = df.isna().sum()
print(empty[empty > 0] if (empty > 0).any() else "No empty values")
print()
print("Empty/blank text in subject/body/answer:")
for c in ["subject", "body", "answer"]:
    print(c, "->", (df[c].isna() | (df[c].astype(str).str.strip() == "")).sum())
print()
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate bodies:", df.duplicated(subset="body").sum())

subject     2607
answer         3
tag_2          6
tag_3         69
tag_4       1711
tag_5       7922
tag_6      12968
tag_7      15204
tag_8      16057
dtype: int64

Empty/blank text in subject/body/answer:
subject -> 2607
body -> 0
answer -> 3

Fully duplicate rows: 0
Duplicate bodies: 0


## 4. Value counts: queue, priority, type

In [6]:
for c in ["queue", "priority", "type"]:
    print(df[c].value_counts(dropna=False), "\n")

queue
Technical Support                  4737
Product Support                    3073
Customer Service                   2410
IT Support                         1942
Billing and Payments               1595
Returns and Exchanges               820
Service Outages and Maintenance     664
Sales and Pre-Sales                 513
Human Resources                     348
General Inquiry                     236
Name: count, dtype: int64 

priority
medium    6618
high      6346
low       3374
Name: count, dtype: int64 

type
Incident    6571
Request     4665
Problem     3397
Change      1705
Name: count, dtype: int64 



## 5. Word counts for body and answer

In [7]:
df["body_words"] = df["body"].fillna("").str.split().str.len()
df["answer_words"] = df["answer"].fillna("").str.split().str.len()
df[["body_words", "answer_words"]].describe().round(1)

,body_words,answer_words
count,16338.0,16338.0
mean,53.9,56.5
std,26.9,25.5
min,1.0,0.0
25%,31.0,34.0
50%,55.0,59.0
75%,78.0,79.0
max,168.0,135.0


## 6. Three full example tickets

In [8]:
for _, r in df.sample(3, random_state=7).iterrows():
    print("SUBJECT :", r["subject"])
    print("QUEUE   :", r["queue"], "| PRIORITY:", r["priority"])
    print("BODY    :", r["body"])
    print("ANSWER  :", r["answer"])
    print("-" * 80)

SUBJECT : Digital Campaign Not Meeting Expectations
QUEUE   : Product Support | PRIORITY: high
BODY    : Contacting customer support about the marketing agency's digital campaign, which is underperforming despite recent strategy modifications. After conducting an audience analysis and updating the ad creatives, there has been little to no improvement. We suspect the problem may be related to targeting errors or changes in the platform's algorithm. We would highly appreciate any assistance or guidance to help us identify and resolve the issue.
ANSWER  : We will review the case and get in touch soon to discuss potential solutions.
--------------------------------------------------------------------------------
SUBJECT : Enhance Project Management Software
QUEUE   : Customer Service | PRIORITY: medium
BODY    : Incorporate WhatsApp real-time updates for better collaboration and feature improvements in the project management software.
ANSWER  : We value your suggestion to integrate WhatsAp

## 7. Ten random tickets with their queue (judge the labels yourself)

In [9]:
for _, r in df.sample(10, random_state=42).iterrows():
    print(f"[{r['queue']}]  {r['subject']}")
    print("   ", str(r["body"])[:300].replace("\n", " "))
    print()

[IT Support]  Query on Data Analytics Tools for Investment Optimization
    Dear Customer Support, I am inquiring about the data analytics tools that are available to assist in optimizing investment strategies. Could you provide detailed information on the types of tools that are used to make informed investment decisions? I would greatly appreciate any information provided

[IT Support]  Problems with Connection
    Customers are facing occasional connectivity problems with the SaaS platform. This could be due to network instability or incorrect configuration. After restarting devices and reconfiguring settings, as well as checking service statuses, no solution has been found. The team is working to resolve thi

[Billing and Payments]  nan
    Hello Customer Support, I am inquiring about optimizing my investments using the data analytics tools available on IBM Cloud. Could you provide details on how to leverage these tools to make informed investment decisions? I am particularly inter